# Buckaroo + xorq — push-down stat pipeline

`XorqStatPipeline` runs buckaroo's summary-stat pipeline against a
**xorq expression** (xorq's vendored ibis) instead of pulling the
table into pandas first. Stats compile to a single batched
`table.aggregate(...)` query plus per-column histogram queries —
computation is pushed to the backend (DuckDB, Postgres, Snowflake,
DataFusion, …) via xorq.

Install the optional dep with `pip install 'buckaroo[xorq]'`.

In [ ]:
import pandas as pd
import xorq.api as xo

from buckaroo.pluggable_analysis_framework.xorq_stat_pipeline import XorqStatPipeline
from buckaroo.pluggable_analysis_framework.df_stats_v2 import XorqDfStatsV2
from buckaroo.pluggable_analysis_framework.stat_func import stat, XorqColumn
from buckaroo.customizations.xorq_stats_v2 import XORQ_STATS_V2

## 1. A xorq table from a pandas DataFrame

`xo.memtable` wraps a pandas/arrow source as a xorq expression
backed by DataFusion (no extra install required). Any other xorq
backend works the same way — pass a `Table` from
`xo.read_parquet`, `xo.read_postgres`, `xo.connect('duckdb://…')`,
etc.

In [ ]:
df = pd.DataFrame({
    'price': [12.5, 18.9, 7.4, 22.1, None, 14.0, 9.9, 31.2, 11.5, 19.8],
    'qty': [1, 2, 1, 3, 1, 2, 4, 1, 2, 5],
    'category': ['a', 'b', 'a', 'c', 'a', 'b', 'b', 'c', 'a', 'b'],
    'is_promo': [True, False, False, True, False, False, True, True, False, True],
})
table = xo.memtable(df)
table.schema()

## 2. Run the pipeline

`XorqStatPipeline(stat_funcs).process_table(table)` returns
`(summary_dict, errors)`. Each column gets a flat dict of stats —
the same shape that `DfStatsV2.sdf` produces for pandas/polars.

In [ ]:
pipeline = XorqStatPipeline(XORQ_STATS_V2)
summary, errors = pipeline.process_table(table)
assert errors == []
pd.DataFrame.from_dict(summary, orient='index')[
    ['_type', 'length', 'null_count', 'distinct_count', 'min', 'max', 'mean', 'std']
]

## 3. What query did we actually run?

Every `@stat` function whose only input is an `XorqColumn` returns
an ibis expression — the pipeline folds them all into a single
`table.aggregate(...)` call. One round-trip to the backend per
table, no matter how many columns or stats.

In [ ]:
# Reconstruct the same batch query the pipeline builds, for inspection.
agg_exprs = {
    'price__null_count': table.price.isnull().sum().coalesce(0).cast('int64'),
    'price__min': table.price.min().cast('float64'),
    'price__max': table.price.max().cast('float64'),
    'price__mean': table.price.mean().cast('float64'),
    'qty__null_count': table.qty.isnull().sum().coalesce(0).cast('int64'),
    'qty__min': table.qty.min().cast('float64'),
    'qty__max': table.qty.max().cast('float64'),
    'qty__mean': table.qty.mean().cast('float64'),
    'category__distinct_count': table.category.nunique().cast('int64'),
    'category__null_count': table.category.isnull().sum().coalesce(0).cast('int64'),
}
print(xo.to_sql(table.aggregate(**agg_exprs)))

## 4. Histograms

Histograms can't be folded into the batch — each column needs its
own `GROUP BY` query. The pipeline runs them in a second phase,
still on the backend (no pulling rows into Python). Numeric
columns get 10 equal-width buckets; non-numeric columns get the
top-10 categories.

In [ ]:
summary['price']['histogram']

In [ ]:
summary['category']['histogram']

## 5. DfStats wrapper for DataFlow integration

`XorqDfStatsV2` mirrors the `DfStatsV2` / `PlDfStatsV2` interface
(`.sdf`, `.errs`, `.add_analysis`) so DataFlow,
`CustomizableDataflow`, and any DfStats consumer can plug a xorq
table in unchanged.

In [ ]:
stats = XorqDfStatsV2(table, XORQ_STATS_V2)
pd.DataFrame.from_dict(stats.sdf, orient='index')[
    ['_type', 'length', 'null_count', 'distinct_count', 'mean', 'std', 'distinct_per']
]

## 6. Adding a custom stat

Any `@stat`-decorated function with an `XorqColumn` parameter
joins the batch aggregate. Return an ibis expression; the pipeline
compiles, runs, and writes the scalar result into the per-column
accumulator under the name in `provides=`.

In [ ]:
@stat(provides='range', column_filter=lambda dt: dt.is_numeric())
def col_range(col: XorqColumn) -> float:
    return (col.max() - col.min()).cast('float64')

@stat(provides='cv', column_filter=lambda dt: dt.is_numeric() and not dt.is_boolean())
def coefficient_of_variation(mean: float, std: float) -> float:
    if mean is None or std is None or mean == 0:
        return None
    return std / mean

extended = [*XORQ_STATS_V2, col_range, coefficient_of_variation]
summary, _ = XorqStatPipeline(extended).process_table(table)
pd.DataFrame.from_dict(summary, orient='index')[['_type', 'min', 'max', 'range', 'mean', 'std', 'cv']]

## 7. Bring your own backend

Pass `backend=` to route every query (batched aggregate **and**
per-column histograms) through a specific xorq backend. Useful
when the table you have is unbound, or when you want to force
execution on a particular engine.

```python
con = xo.connect('duckdb://')          # or postgres, snowflake, ...
table = con.read_parquet('s3://.../events.parquet')
summary, _ = XorqStatPipeline(XORQ_STATS_V2, backend=con).process_table(table)
```

The aggregate and histogram queries run on `con`; only the small
scalar results come back to Python.